In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.landing")
for s in ["bronze", "silver", "gold", "ops"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{s}")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.landing.raw")

base = "/Volumes/workspace/landing/raw"
for p in ["transactions", "customers_cdc", "card_events", "_checkpoints", "_schemas"]:
    dbutils.fs.mkdirs(f"{base}/{p}")
display(dbutils.fs.ls(base))

In [0]:
import json, random, uuid, csv
from datetime import datetime, timedelta

BASE = "/Volumes/workspace/landing/raw"
NUM_CUSTOMERS = 200

def ts(days_back=3):
    return datetime.utcnow() - timedelta(minutes=random.randint(0, days_back * 1440))

# 1. Transactions (like CSV files dropped in S3)
def write_transactions_batch(n=500):
    path = f"{BASE}/transactions/txn_{datetime.utcnow():%Y%m%d_%H%M%S}.csv"
    rows = []
    for _ in range(n):
        t = ts()
        date_str = t.strftime("%Y-%m-%d %H:%M:%S") if random.random() > 0.1 else t.strftime("%d/%m/%Y %H:%M")
        amount = round(random.uniform(-2000, 5000), 2)
        if random.random() < 0.03:
            amount = ""
        rows.append({
            "txn_id": str(uuid.uuid4()),
            "account_id": f"ACC{random.randint(1, NUM_CUSTOMERS):05d}",
            "txn_ts": date_str,
            "amount": amount,
            "currency": random.choice(["USD", "USD", "USD", "EUR", "usd"]),
            "merchant": random.choice(["Amazon", "Walmart", "Shell", "Uber", "Payroll ACME", "Rent LLC", "Starbucks"]),
            "channel": random.choice(["card", "ach", "wire", "atm"]),
        })
    rows += random.sample(rows, k=max(1, n // 50))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)
    print("wrote", path, len(rows))

# 2. Customer changes (like Postgres -> AWS DMS)
def write_customer_cdc_batch(first_load=False):
    path = f"{BASE}/customers_cdc/cdc_{datetime.utcnow():%Y%m%d_%H%M%S}.json"
    events = []
    ids = range(1, NUM_CUSTOMERS + 1) if first_load else random.sample(range(1, NUM_CUSTOMERS + 1), 20)
    for i in ids:
        op = "c" if first_load else random.choice(["u", "u", "u", "d"])
        after = None if op == "d" else {
            "customer_id": i,
            "full_name": f"Customer {i}",
            "email": f"customer{i}@example.com",
            "ssn_last4": f"{random.randint(0, 9999):04d}",
            "city": random.choice(["Dallas", "Austin", "Chicago", "NYC", "Seattle"]),
            "risk_tier": random.choice(["low", "medium", "high"]),
        }
        events.append({"op": op, "customer_id": i, "after": after,
                       "ts_ms": int(datetime.utcnow().timestamp() * 1000) + random.randint(0, 5000)})
    with open(path, "w") as f:
        for e in events:
            f.write(json.dumps(e) + "\n")
    print("wrote", path, len(events))

# 3. Card events (like Kinesis -> Firehose -> S3)
def write_card_events_batch(n=300, drift=False):
    path = f"{BASE}/card_events/events_{datetime.utcnow():%Y%m%d_%H%M%S}.json"
    with open(path, "w") as f:
        for _ in range(n):
            e = {"event_id": str(uuid.uuid4()),
                 "card_id": f"CARD{random.randint(1, 400):05d}",
                 "account_id": f"ACC{random.randint(1, NUM_CUSTOMERS):05d}",
                 "event_type": random.choice(["auth", "auth", "decline", "pin_fail"]),
                 "amount": round(random.uniform(1, 3000), 2),
                 "country": random.choice(["US"] * 9 + ["RU", "NG", "BR"]),
                 "event_ts": ts(1).isoformat()}
            if drift:
                e["device_type"] = random.choice(["mobile", "pos", "web"])
            f.write(json.dumps(e) + "\n")
    print("wrote", path, n)

In [0]:
# First run only: loads all 200 customers, plus one batch of transactions and card events
write_customer_cdc_batch(first_load=True)
write_transactions_batch()
write_card_events_batch()

In [0]:
display(spark.read.option("header", True).csv(f"{BASE}/transactions"))

In [0]:
display(spark.read.json(f"{BASE}/customers_cdc").limit(5))